# ☕ Dashboard Kopi Indonesia — Google Colab
**Data Preprocessing + Visualisasi + Deploy ke Streamlit**

Dataset: Produksi, Luas Areal, dan Produktivitas Kopi per Provinsi (2014–2021)

---

## 📦 1. Install Dependencies

In [ ]:
!pip install streamlit pyngrok xlrd openpyxl plotly -q

## 📁 2. Upload File Data

> **Upload 3 file berikut:** `Produksi-Kopi.xls`, `Areal-Kopi.xls`, `Prodtv-Kopi.xls`

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
os.makedirs('/content/data', exist_ok=True)
for fname in uploaded:
    os.rename(f'/content/{fname}', f'/content/data/{fname}')
print('File terupload:', os.listdir('/content/data'))

## 🔍 3. Data Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

years = [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]

def parse_file(path, row_start=10, row_end=44):
    df = pd.read_excel(path, engine='xlrd', header=None)
    data = df.iloc[row_start:row_end, [1, 2, 3, 4, 5, 6, 7, 8, 9]].copy()
    data.columns = ['Provinsi'] + years
    data = data[data['Provinsi'].notna()].copy()
    data['Provinsi'] = data['Provinsi'].astype(str).str.strip()
    for y in years:
        data[y] = pd.to_numeric(data[y], errors='coerce')
    data = data[data['Provinsi'].str.lower() != 'indonesia']
    data = data.dropna(subset=['Provinsi'])
    data = data[~data['Provinsi'].str.startswith('**')]
    data = data[~data['Provinsi'].str.startswith('-)']]
    return data.reset_index(drop=True)

produksi = parse_file('/content/data/Produksi-Kopi.xls', row_start=11, row_end=45)
areal    = parse_file('/content/data/Areal-Kopi.xls',    row_start=10, row_end=44)
prodtv   = parse_file('/content/data/Prodtv-Kopi.xls',   row_start=10, row_end=44)

print('✅ Produksi shape:', produksi.shape)
print('✅ Areal shape:', areal.shape)
print('✅ Produktivitas shape:', prodtv.shape)

In [ ]:
# Melt ke long format
def to_long(df, val_name):
    return df.melt(id_vars='Provinsi', value_vars=years, var_name='Tahun', value_name=val_name)

long_prod = to_long(produksi, 'Produksi_Ton')
long_area = to_long(areal,    'Luas_Areal_Ha')
long_prtv = to_long(prodtv,   'Produktivitas_KgHa')

# Merge
merged = long_prod.merge(long_area, on=['Provinsi','Tahun']).merge(long_prtv, on=['Provinsi','Tahun'])
merged['Tahun'] = merged['Tahun'].astype(int)

# Region mapping
region_map = {
    'Aceh':'Sumatera','Sumatera Utara':'Sumatera','Sumatera Barat':'Sumatera',
    'Riau':'Sumatera','Kepulauan Riau':'Sumatera','Jambi':'Sumatera',
    'Sumatera Selatan':'Sumatera','Kepulauan Bangka Belitung':'Sumatera',
    'Bengkulu':'Sumatera','Lampung':'Sumatera',
    'DKI Jakarta':'Jawa','Jawa Barat':'Jawa','Banten':'Jawa',
    'Jawa Tengah':'Jawa','DI. Yogyakarta':'Jawa','Jawa Timur':'Jawa',
    'Bali':'Bali & Nusa Tenggara','Nusa Tenggara Barat':'Bali & Nusa Tenggara',
    'Nusa Tenggara Timur':'Bali & Nusa Tenggara',
    'Kalimantan Barat':'Kalimantan','Kalimantan Tengah':'Kalimantan',
    'Kalimantan Selatan':'Kalimantan','Kalimantan Timur':'Kalimantan','Kalimantan Utara':'Kalimantan',
    'Sulawesi Utara':'Sulawesi','Gorontalo':'Sulawesi','Sulawesi Tengah':'Sulawesi',
    'Sulawesi Selatan':'Sulawesi','Sulawesi Barat':'Sulawesi','Sulawesi Tenggara':'Sulawesi',
    'Maluku':'Maluku & Papua','Maluku Utara':'Maluku & Papua','Papua':'Maluku & Papua','Papua Barat':'Maluku & Papua',
}
merged['Wilayah'] = merged['Provinsi'].map(region_map).fillna('Lainnya')

# Interpolasi missing values
for col in ['Produksi_Ton','Luas_Areal_Ha','Produktivitas_KgHa']:
    merged[col] = merged.groupby('Provinsi')[col].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both'))

print('✅ Dataset final shape:', merged.shape)
print('Missing values:')
print(merged.isnull().sum())
merged.head(10)

## 📊 4. Statistik Deskriptif

In [ ]:
print('=== Statistik Deskriptif ===')
merged[['Produksi_Ton','Luas_Areal_Ha','Produktivitas_KgHa']].describe().round(2)

## 📈 5. Visualisasi dengan Plotly

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Tren Nasional
nasional = merged.groupby('Tahun')[['Produksi_Ton','Luas_Areal_Ha']].sum().reset_index()
nasional['Produktivitas_KgHa'] = merged.groupby('Tahun')['Produktivitas_KgHa'].mean().values

fig = make_subplots(rows=1, cols=3, subplot_titles=('Produksi (Ton)','Luas Areal (Ha)','Produktivitas (Kg/Ha)'))
for i,(col,color) in enumerate([('Produksi_Ton','#8B4513'),('Luas_Areal_Ha','#228B22'),('Produktivitas_KgHa','#DAA520')],1):
    fig.add_trace(go.Scatter(x=nasional['Tahun'],y=nasional[col],mode='lines+markers',
                             line=dict(color=color,width=3),marker=dict(size=8),name=col),row=1,col=i)
fig.update_layout(title='Tren Kopi Indonesia 2014-2021',height=350,showlegend=False)
fig.show()

In [ ]:
# Top 10 Provinsi
top10 = merged[merged['Tahun']==2021].groupby('Provinsi')['Produksi_Ton'].sum().nlargest(10).reset_index()
fig2 = px.bar(top10, x='Produksi_Ton', y='Provinsi', orientation='h',
              color='Produksi_Ton', color_continuous_scale='YlOrBr',
              title='Top 10 Provinsi Produksi Kopi 2021',
              labels={'Produksi_Ton':'Produksi (Ton)'})
fig2.show()

In [ ]:
# Heatmap
pivot = merged.pivot_table(values='Produksi_Ton', index='Provinsi', columns='Tahun', aggfunc='sum')
fig3 = px.imshow(pivot, color_continuous_scale='YlOrBr', aspect='auto',
                 title='Heatmap Produksi per Provinsi per Tahun',
                 labels={'color':'Produksi (Ton)'})
fig3.show()

In [ ]:
# Korelasi
corr = merged[['Produksi_Ton','Luas_Areal_Ha','Produktivitas_KgHa']].corr()
fig4 = px.imshow(corr, text_auto=True, color_continuous_scale='RdBu_r',
                 zmin=-1, zmax=1, title='Korelasi Antar Variabel')
fig4.show()

In [ ]:
# Simpan CSV
merged.to_csv('/content/kopi_indonesia_clean.csv', index=False)
print('✅ Data disimpan ke kopi_indonesia_clean.csv')
files.download('/content/kopi_indonesia_clean.csv')

## 🚀 6. Deploy Streamlit via ngrok

> **Cara:** Buat akun gratis di https://ngrok.com → copy authtoken → paste di bawah

In [ ]:
# Tulis file app.py ke Colab
app_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

st.set_page_config(page_title="Dashboard Kopi Indonesia", page_icon="☕", layout="wide")

@st.cache_data
def load_data():
    years = [2014,2015,2016,2017,2018,2019,2020,2021]
    def parse_file(path, row_start=10, row_end=44):
        df = pd.read_excel(path, engine="xlrd", header=None)
        data = df.iloc[row_start:row_end, [1,2,3,4,5,6,7,8,9]].copy()
        data.columns = ["Provinsi"] + years
        data = data[data["Provinsi"].notna()].copy()
        data["Provinsi"] = data["Provinsi"].astype(str).str.strip()
        for y in years:
            data[y] = pd.to_numeric(data[y], errors="coerce")
        data = data[data["Provinsi"].str.lower() != "indonesia"]
        data = data.dropna(subset=["Provinsi"])
        data = data[~data["Provinsi"].str.startswith("**")]
        return data.reset_index(drop=True)
    p = parse_file("/content/data/Produksi-Kopi.xls", 11, 45)
    a = parse_file("/content/data/Areal-Kopi.xls", 10, 44)
    t = parse_file("/content/data/Prodtv-Kopi.xls", 10, 44)
    def melt(df, v): return df.melt(id_vars="Provinsi", value_vars=years, var_name="Tahun", value_name=v)
    m = melt(p,"Produksi_Ton").merge(melt(a,"Luas_Areal_Ha"),on=["Provinsi","Tahun"]).merge(melt(t,"Produktivitas_KgHa"),on=["Provinsi","Tahun"])
    m["Tahun"] = m["Tahun"].astype(int)
    rmap = {"Aceh":"Sumatera","Sumatera Utara":"Sumatera","Sumatera Barat":"Sumatera","Riau":"Sumatera","Kepulauan Riau":"Sumatera","Jambi":"Sumatera","Sumatera Selatan":"Sumatera","Kepulauan Bangka Belitung":"Sumatera","Bengkulu":"Sumatera","Lampung":"Sumatera","DKI Jakarta":"Jawa","Jawa Barat":"Jawa","Banten":"Jawa","Jawa Tengah":"Jawa","DI. Yogyakarta":"Jawa","Jawa Timur":"Jawa","Bali":"Bali & Nusa Tenggara","Nusa Tenggara Barat":"Bali & Nusa Tenggara","Nusa Tenggara Timur":"Bali & Nusa Tenggara","Kalimantan Barat":"Kalimantan","Kalimantan Tengah":"Kalimantan","Kalimantan Selatan":"Kalimantan","Kalimantan Timur":"Kalimantan","Kalimantan Utara":"Kalimantan","Sulawesi Utara":"Sulawesi","Gorontalo":"Sulawesi","Sulawesi Tengah":"Sulawesi","Sulawesi Selatan":"Sulawesi","Sulawesi Barat":"Sulawesi","Sulawesi Tenggara":"Sulawesi","Maluku":"Maluku & Papua","Maluku Utara":"Maluku & Papua","Papua":"Maluku & Papua","Papua Barat":"Maluku & Papua"}
    m["Wilayah"] = m["Provinsi"].map(rmap).fillna("Lainnya")
    for c in ["Produksi_Ton","Luas_Areal_Ha","Produktivitas_KgHa"]:
        m[c] = m.groupby("Provinsi")[c].transform(lambda x: x.interpolate(method="linear",limit_direction="both"))
    return m

df = load_data()
st.title("☕ Dashboard Kopi Indonesia 2014-2021")
st.sidebar.markdown("## Filter")
yr = st.sidebar.slider("Tahun", 2014, 2021, (2014, 2021))
regions = st.sidebar.multiselect("Wilayah", sorted(df["Wilayah"].unique()), default=sorted(df["Wilayah"].unique()))
dff = df[(df["Tahun"].between(yr[0],yr[1])) & (df["Wilayah"].isin(regions))]

c1,c2,c3 = st.columns(3)
c1.metric("Total Produksi (Ton)", f"{dff[dff.Tahun==dff.Tahun.max()]["Produksi_Ton"].sum():,.0f}")
c2.metric("Total Luas Areal (Ha)", f"{dff[dff.Tahun==dff.Tahun.max()]["Luas_Areal_Ha"].sum():,.0f}")
c3.metric("Rerata Produktivitas", f"{dff[dff.Tahun==dff.Tahun.max()]["Produktivitas_KgHa"].mean():,.1f} Kg/Ha")

tab1,tab2,tab3 = st.tabs(["Tren Waktu","Per Provinsi","Data"])
with tab1:
    nas = dff.groupby("Tahun")[["Produksi_Ton","Luas_Areal_Ha"]].sum().reset_index()
    fig = px.line(nas, x="Tahun", y="Produksi_Ton", markers=True, title="Tren Produksi Nasional")
    st.plotly_chart(fig, use_container_width=True)
with tab2:
    yr2 = st.selectbox("Tahun", sorted(dff.Tahun.unique(), reverse=True))
    fig2 = px.bar(dff[dff.Tahun==yr2].sort_values("Produksi_Ton"), y="Provinsi", x="Produksi_Ton",
                  orientation="h", color="Produksi_Ton", color_continuous_scale="YlOrBr", height=600)
    st.plotly_chart(fig2, use_container_width=True)
with tab3:
    st.dataframe(dff, use_container_width=True)
    st.download_button("Download CSV", dff.to_csv(index=False), "kopi_clean.csv")
'''
with open('/content/app.py', 'w') as f:
    f.write(app_code)
print('✅ app.py ditulis')

In [ ]:
# ⚠️ GANTI dengan authtoken Anda dari https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = 'PASTE_TOKEN_NGROK_ANDA_DISINI'

!pip install pyngrok -q
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)

import subprocess, time, threading

def run_app():
    subprocess.run(['streamlit', 'run', '/content/app.py',
                    '--server.port=8501', '--server.headless=true',
                    '--server.enableCORS=false'])

t = threading.Thread(target=run_app, daemon=True)
t.start()
time.sleep(5)

public_url = ngrok.connect(8501)
print('\n' + '='*60)
print('🚀 STREAMLIT RUNNING!')
print(f'🌐 URL Publik: {public_url}')
print('='*60)

---
## 🌐 7. Deploy ke Streamlit Cloud (Permanen)

Untuk deploy **permanen gratis** di Streamlit Community Cloud:

1. **Buat repo GitHub** → upload `app.py`, `requirements.txt`, folder `data/`
2. **Buka** https://share.streamlit.io
3. **Connect GitHub** → pilih repo → klik **Deploy!**
4. App online dalam 2-3 menit ✅

```
struktur_repo/
├── app.py
├── requirements.txt
└── data/
    ├── Produksi-Kopi.xls
    ├── Areal-Kopi.xls
    └── Prodtv-Kopi.xls
```